# 04 Transition Behavior
Question: What happens to strategy behavior around historical macro regime transitions?

**Warning:** This notebook is descriptive only and must not be used for forecasting, signal generation, or trading decisions.

Transition analysis in this notebook is historical context and stress analysis only. It aligns realized strategy returns to realized macro transition dates to describe what happened around those dates; it does not imply prediction, timing, or any forward-looking signal.


In [ ]:
from pathlib import Path
import sys

def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing 'src' and 'data'.")

REPO_ROOT = _repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.conditioning.regime_join import join_returns_with_regimes
from src.conditioning.transition_analysis import build_transition_windows, summarize_transition_windows
from src.data.loaders import load_regimes, load_strategy_returns, load_transitions
from src.data.validators import run_all_validations
from src.visualization.plots import plot_transition_profile

regimes = load_regimes(REPO_ROOT / "data" / "macro" / "regimes_example.csv")
transitions = load_transitions(REPO_ROOT / "data" / "macro" / "transitions_example.csv")
strategy_returns = load_strategy_returns(REPO_ROOT / "data" / "strategies" / "strategy_returns_example.csv")

# Validation is required before any transition-aligned diagnostics.
run_all_validations(regimes, transitions, strategy_returns)

joined = join_returns_with_regimes(strategy_returns, regimes)
event_windows = build_transition_windows(joined, transitions, window=3)
transition_summary = summarize_transition_windows(event_windows).sort_values(["strategy_id", "relative_month"]).reset_index(drop=True)

aligned_transitions = event_windows[["transition_date", "from", "to"]].drop_duplicates().reset_index(drop=True)
transition_overview = {
    "provided_transition_count": len(transitions),
    "aligned_transition_count": len(aligned_transitions),
    "aligned_transition_types": aligned_transitions[["from", "to"]].drop_duplicates().shape[0],
}
display(transition_overview)
display(aligned_transitions)
display(event_windows.sort_values(["transition_date", "strategy_id", "relative_month"]).reset_index(drop=True))
display(transition_summary)

fig = plot_transition_profile(transition_summary, strategy_id="strat_a")
fig
